# OpenAQ data acquisition
MATH70076 Assessment 2 — Question 1

Run the cells in order. Run `pip install -e ".[dev]"` in the project root first.


## 0. Dependencies

Run once. If you already have these, skip it.

In [ ]:
%pip install -e "..[dev]"

## 1. Import the module

The acquisition logic lives in `src/openaq_survey/client.py` rather than in this notebook,
so it can be tested, reused and cited as evidence. This notebook is the
narrative; the module is the tool.

`%autoreload` means edits to the .py take effect without restarting the kernel.

In [ ]:
%load_ext autoreload
%autoreload 2

import json
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

from openaq_survey import client as oaq

DATA = Path("..") / "data"
DATA.mkdir(exist_ok=True)
print("ready")

## 2. API key

Running the next cell pops up a hidden input box. Paste your key and press Enter.

The key is held in memory only — it is **not** written into this notebook file,
so the notebook is safe to commit to a public repo.

In [ ]:
oaq.set_api_key()

## 3. Check the connection

A known station (New Delhi, id 8118) from the OpenAQ quick-start docs.

In [ ]:
probe = oaq.get("/locations/8118")
print(json.dumps(probe["results"][0], indent=2, ensure_ascii=False)[:1200])

## 4. What pollutants exist?

Gives the numeric `parameter_id` values used elsewhere in the API.

In [ ]:
params = oaq.get("/parameters", {"limit": 100})["results"]
pd.DataFrame(params)[["id", "name", "units", "displayName"]].sort_values("id")

## 5. Fetch the station inventory

This sweeps every monitoring station on the platform. **Expect several minutes.**

The result is cached to `data/raw_locations.json`. Re-running this cell after the
first time reads the cache and makes no network calls, which is what lets the
analysis be reproduced offline.

If you see `[429] rate limited` lines — screenshot them. That is direct evidence
for the reflection on rate limits.

In [ ]:
records = oaq.cached_json(
    DATA / "raw_locations.json",
    lambda: list(oaq.paginate("/locations")),
)
print(f"\n{len(records):,} station records")

### Inspect one raw record

Before flattening, look at what the API actually returns. If the flattening in
step 6 produces all-null columns, the field names differ from what was assumed —
this cell is where you find that out.

In [ ]:
print(json.dumps(records[0], indent=2, ensure_ascii=False))

## 6. Flatten to tidy tables

- `stations` — one row per monitoring station
- `sensors` — one row per station × pollutant

In [ ]:
stations = oaq.flatten_stations(records)
sensors  = oaq.flatten_sensors(records)

stations.to_csv(DATA / "stations.csv", index=False)
sensors.to_csv(DATA / "sensors.csv", index=False)

print(f"stations: {stations.shape}   sensors: {sensors.shape}")
stations.head()

### Did anything fail to parse?

Any column at 1.00 means the field name guess was wrong — check the raw record above.

In [ ]:
stations.isna().mean().sort_values(ascending=False).to_frame("null_fraction")

## 7. Summary

These are the numbers that determine what argument the data can support.

In [ ]:
print(f"stations        : {len(stations):,}")
print(f"sensors         : {len(sensors):,}")
print(f"countries       : {stations['country_code'].nunique()}")
print(f"earliest record : {stations['datetime_first'].min()}")
print(f"latest record   : {stations['datetime_last'].max()}")

print("\nreference monitors vs other:")
print(stations["is_monitor"].value_counts(dropna=False))

print("\ntop 15 countries by station count:")
print(stations["country_code"].value_counts().head(15))

print("\nfewest stations:")
print(stations["country_code"].value_counts().tail(15))

print("\npollutant coverage:")
print(sensors["parameter"].value_counts().head(15))

print("\nlicences:")
print(stations["licence"].value_counts(dropna=False).head(10))

print("\ndata providers:")
print(stations["provider"].value_counts().head(10))

## 8. Per-country table

The unit of analysis for the argument about monitoring coverage.

In [ ]:
by_country = (
    stations
    .groupby(["country_code", "country_name"], dropna=False)
    .agg(
        n_stations=("location_id", "count"),
        n_reference=("is_monitor", "sum"),
        n_sensors=("n_sensors", "sum"),
        earliest=("datetime_first", "min"),
    )
    .reset_index()
    .sort_values("n_stations", ascending=False)
)
by_country.to_csv(DATA / "by_country.csv", index=False)
print(f"{len(by_country)} countries")
by_country.head(30)

In [ ]:
from openaq_survey import worldbank as wb

countries = wb.fetch_countries()
pop   = wb.latest_non_null(wb.fetch_indicator(wb.POPULATION))
pm25  = wb.latest_non_null(wb.fetch_indicator(wb.PM25_EXPOSURE))
gni   = wb.latest_non_null(wb.fetch_indicator(wb.GNI_PER_CAPITA))

print(countries.shape, pop.shape, pm25.shape, gni.shape)
countries.head()

In [ ]:
from openaq_survey import combine

wb = combine.prepare_worldbank(
    countries,
    {"population": pop, "pm25": pm25, "gni_per_capita": gni},
)

diag = combine.diagnose_join(by_country, wb)

table = combine.build_country_table(by_country, wb)
table.to_csv(DATA / "country_table.csv", index=False)

In [ ]:

diag["openaq_only"][["country_code", "country_name", "n_stations"]]


(diag["worldbank_only"][["country_name", "region", "income_group", "population"]]
 .sort_values("population", ascending=False)
 .head(25))

In [ ]:
cols = ["country_name", "region", "income_group", "n_stations",
        "n_reference", "reference_per_million", "ref_share", "pm25"]

sub = table[table["population"] > 5e6]
print(sub.nsmallest(20, "reference_per_million")[cols].to_string(index=False))
print(sub.nlargest(20, "reference_per_million")[cols].to_string(index=False))

## 9. Before committing

Notebook outputs are saved inside the `.ipynb` file. Clear them before pushing,
or install `nbstripout` to do it automatically on every commit:

```
pip install nbstripout
nbstripout --install
```

Keep a note as you go of anything that broke, surprised you, or made you change
approach — that is the raw material for the reflective writing, and it will not
be recoverable from memory in three days.


In [ ]:


# --- data ---
plot_df = table[(table["population"] > 5e6) & table["pm25"].notna()].copy()

# Among countries with zero reference-grade monitors, show the 10 with the
# highest PM2.5 exposure; ties broken by population (more people affected first).
worst = plot_df[plot_df["reference_per_million"] == 0].nlargest(10, ["pm25", "population"])
best  = plot_df.nlargest(10, "reference_per_million")
combined = pd.concat([worst, best])

labels  = combined["country_name"].tolist()
values  = combined["reference_per_million"].to_numpy()
colours = combined["pm25"].to_numpy()

# --- figure ---
fig, ax = plt.subplots(figsize=(9, 8))
y = np.arange(len(combined))
norm = plt.Normalize(colours.min(), colours.max())
cmap = plt.cm.YlOrBr

bars = ax.barh(y, values, color=cmap(norm(colours)), edgecolor="grey", linewidth=0.4)
ax.set_yticks(y)
ax.set_yticklabels(labels)
ax.invert_yaxis()
ax.axhline(len(worst) - 0.5, color="grey", linewidth=0.8)

# Zero-length bars are invisible: mark them with a PM2.5-coloured dot and
# annotate the PM2.5 value so the colour encoding still carries information.
xmax = values.max()
for yi, v, c in zip(y, values, colours):
    if v == 0:
        ax.scatter(0, yi, s=45, color=cmap(norm(c)), edgecolor="grey",
                   linewidth=0.4, zorder=3, clip_on=False)
        ax.annotate(f"0 monitors · PM2.5 {c:.0f} µg/m³", (0, yi),
                    xytext=(8, 0), textcoords="offset points",
                    va="center", fontsize=8, color="dimgrey")
    else:
        ax.annotate(f"{v:.1f}", (v, yi), xytext=(4, 0),
                    textcoords="offset points", va="center", fontsize=8)

# Group labels so the dividing line reads without a legend
ax.text(xmax, (len(worst) - 1) / 2, "Highest PM2.5,\nno reference monitors",
        ha="right", va="center", fontsize=9, style="italic", color="dimgrey")
ax.text(xmax, len(worst) + (len(best) - 1) / 2, "Best-covered countries",
        ha="right", va="center", fontsize=9, style="italic", color="dimgrey")

ax.set_xlabel("Reference-grade monitors per million people (OpenAQ)")
ax.set_title("The most polluted air is the least monitored:\n"
             "reference-grade coverage vs PM2.5 exposure, countries > 5 M people")

sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
cbar = fig.colorbar(sm, ax=ax)
cbar.set_label("Population-weighted mean annual PM2.5 exposure (µg/m³)")

fig.text(0.01, 0.02,
         "Sources: OpenAQ v3 locations API; World Bank WDI.\n"
         "Zero counts reflect absence from OpenAQ, not absence of national monitoring.",
         fontsize=7, color="dimgrey")

fig.tight_layout(rect=[0, 0.05, 1, 1])
fig.savefig("../figures/coverage.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
from openaq_survey.distance import nearest_reference_km

coords_ok = stations.dropna(subset=["latitude", "longitude"])
ref     = coords_ok[coords_ok["is_monitor"]]
lowcost = coords_ok[~coords_ok["is_monitor"]]
print(f"dropped {len(stations) - len(coords_ok)} stations with missing coordinates")

d = nearest_reference_km(
    lowcost["latitude"], lowcost["longitude"],
    ref["latitude"], ref["longitude"],
)
print(f"n = {len(d)}")
print(f"median {np.median(d):.1f} km | 90th pct {np.percentile(d, 90):.1f} km | max {d.max():.0f} km")

In [ ]:
import time
from openaq_survey.distance import nearest_reference_km, nearest_reference_km_naive

rng = np.random.default_rng(1)
idx = rng.choice(len(lowcost), size=500, replace=False)
sub = lowcost.iloc[idx]

t0 = time.perf_counter()
d_naive = nearest_reference_km_naive(sub["latitude"], sub["longitude"],
                                     ref["latitude"], ref["longitude"])
t_naive = time.perf_counter() - t0

t0 = time.perf_counter()
d_fast = nearest_reference_km(sub["latitude"], sub["longitude"],
                              ref["latitude"], ref["longitude"])
t_fast = time.perf_counter() - t0

assert np.allclose(d_naive, d_fast)
print(f"naive : {t_naive:.2f} s / 500 queries  -> est. full dataset {t_naive*len(lowcost)/500/60:.1f} min")
print(f"fast  : {t_fast:.3f} s / 500 queries")
print(f"speedup: {t_naive/t_fast:.0f}x")

In [ ]:
%pip install line_profiler
%load_ext line_profiler
%lprun -f nearest_reference_km_naive nearest_reference_km_naive(sub["latitude"][:50], sub["longitude"][:50], ref["latitude"], ref["longitude"])

In [ ]:
%lprun -f nearest_reference_km nearest_reference_km(sub["latitude"], sub["longitude"], ref["latitude"], ref["longitude"])

In [ ]:
for cs in [50, 200, 1000, 4000, len(lowcost)]:
    t0 = time.perf_counter()
    nearest_reference_km(lowcost["latitude"], lowcost["longitude"],
                         ref["latitude"], ref["longitude"], chunk_size=cs)
    mem_mb = cs * len(ref) * 8 / 1e6
    print(f"chunk_size {cs:>5}: {time.perf_counter()-t0:.2f} s   (intermediate ~{mem_mb:,.0f} MB)")